
# Generación del conjunto de datos de entrenamiento

**Autor**: Juan Carlos Alfaro Jiménez

El objetivo de esta libreta es construir el conjunto de datos de entrenamiento para el modelo de detección de fraude en tarjetas de crédito. Para ello, se combina la tabla `gold_fraud_spine` (que contiene el esqueleto de eventos etiquetados) con las dos tablas de características de la capa `Gold` (`gold_customer_profile` y `gold_customer_aggregations`), usando la `API` del `Feature Store` de `Databricks`.

El resultado final se guarda como tabla `Delta` estática en `Unity Catalog` bajo el nombre `gold_fraud_training_dataset`. Esta tabla cumple tres funciones críticas en la arquitectura:

* **Conjunto de datos de entrenamiento reproducible**: el cruce temporal entre la *spine* y las tablas de características (la operación más costosa del *pipeline*) se realiza una única vez. Los experimentos posteriores leen directamente desde esta tabla `Delta`, sin recalcular nada.
* **Fotografía congelada de los datos**: las tablas `Gold` son «vivas» y se actualizan cada hora. Al guardar el conjunto de entrenamiento en `Delta`, la versión exacta de los datos que vio el modelo queda registrada de forma inmutable, lo que permite reproducir cualquier entrenamiento meses después.
* **Referencia de *baseline* para monitorización**: esta misma tabla se usará en las libretas de monitorización como perfil de referencia para detectar *data drift* cuando el modelo esté en producción.

Esta libreta **no contiene lógica de transformación propia**. Todo el trabajo de cruce y enriquecimiento lo realiza internamente `create_training_set`, que delega en `Spark` la ejecución del *point-in-time* (`PiT`) *join* de forma distribuida.


## 1. Importación de librerías y configuración

La primera celda instala el paquete `databricks-feature-engineering`, necesario para acceder a `create_training_set` y a los objetos `FeatureLookup`.

A continuación se importan los componentes necesarios y se definen los nombres completamente cualificados de todas las tablas involucradas:

* `gold_fraud_spine`: la tabla de partida del cruce. Contiene el `customer_id`, el `timestamp` de cada transacción, los detalles de la misma y la etiqueta `is_fraud`.
* `gold_customer_profile`: tabla de características estáticas del cliente (edad, país, segmento, etc.).
* `gold_customer_aggregations`: tabla de características comportamentales calculadas con ventana deslizante (número de transacciones en los últimos 7 días, importe medio, etc.).
* `gold_fraud_training_dataset`: la tabla de salida donde se guardará el conjunto de entrenamiento enriquecido.

También se inicializa el cliente `FeatureEngineeringClient`.

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from datetime import datetime, timezone
from pyspark.sql.functions import col, count, max, round, when

In [0]:
catalog = "workspace"
database = "credit_card_fraud"

gold_spine_table = f"{catalog}.{database}.gold_fraud_spine"
gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

gold_training_dataset_table = f"{catalog}.{database}.gold_fraud_training_dataset"

fe = FeatureEngineeringClient()


## 2. Carga de la *spine*

La *spine* es el punto de partida de todo el proceso. Es la tabla que determina **qué filas** compondrán el conjunto de entrenamiento y **en qué instante** se evalúa cada cruce. Sin la *spine*, la `API` `create_training_set` no sabe ni cuántas filas debe generar ni qué marcas temporales debe usar para el `PiT` *join*.

La *spine* debe contener obligatoriamente:

* La clave de unión con las tablas de características: en nuestro caso, `customer_id`.
* La marca temporal del evento: `timestamp`. Esta columna es la que se pasa como `timestamp_lookup_key` en los `FeatureLookup` y define el `AS OF` de cada cruce.
* La etiqueta de supervisión: `is_fraud`. Esta columna es la variable objetivo que el modelo deberá aprender a predecir.

Se muestra el esquema y una muestra de la *spine* para verificar que tiene la estructura esperada antes de lanzar el cruce.

In [0]:
spine_df = spark.table(gold_spine_table)

print(f"Spine rows: {spine_df.count():,}")
print(f"Spine columns: {len(spine_df.columns)}")
spine_df.printSchema()


## 3. Definición de los `FeatureLookup`

Un `FeatureLookup` es el objeto que le indica a `create_training_set` cómo enriquecer cada fila de la *spine* con características procedentes de una tabla del `Feature Store`. Por cada tabla de características que queramos incorporar, se crea un `FeatureLookup` independiente.

Cada `FeatureLookup` tiene cuatro parámetros fundamentales:

* **`table_name`**: nombre completamente cualificado de la tabla de características en `Unity Catalog`.
* **`feature_names`**: lista de columnas de esa tabla que se quieren añadir al conjunto de datos de entrenamiento. Si se omite este parámetro, se incorporan todas las columnas de la tabla.
* **`lookup_key`**: columna o lista de columnas que se usa como clave de unión primaria entre la *spine* y la tabla de características. En nuestro caso, `customer_id`.
* **`timestamp_lookup_key`**: columna de la *spine* que actúa como referencia temporal. Aquí es donde se ejecuta el `PiT` *join*: se buscará, para cada fila de la *spine*, el registro más reciente de la tabla de características cuyo *timestamp* sea **anterior o igual** al valor de esta columna. Esto garantiza el comportamiento `AS OF` y previene cualquier fuga de datos del futuro (*data leakage*).

> **Nota arquitectónica clave**: Observa en el código posterior que no es necesario indicar a `FeatureLookup` cómo se llama la columna temporal en la tabla de características (por ejemplo, `__START_AT` en `gold_customer_profile`). El `Feature Store` lo sabe automáticamente porque definimos esa columna con el modificador `TIMESERIES` en la clave primaria al crear las tablas `Gold`.

A continuación, definimos dos `FeatureLookup`, uno por cada tabla de características de la capa `Gold`.

In [0]:
entity_key = "customer_id"  # Shared join key: links the transaction to the customer in both tables
timestamp_key = "timestamp"  # Spine timestamp column for the join

# Static or slowly-changing customer profile features
profile_feature_names = [
    # Demographic
    "age",  # Raw age in years
    "age_group", # Bucketed: young, adult, or senior
    "gender",  # M, F, or O
    "occupation",  # Job category
    "city_tier",  # 1, 2, or 3 (urbanization level of the customer)

    # Financial profile
    "income_bracket",  # Declared income band
    "income_group",  # Bucketed: low, medium, or high
    "customer_segment",  # standard, premium, vip, business, or student
    "card_type",  # visa, mastercard, amex, or discover

    # Account security & behaviour
    "num_cards_issued",  # Number of active cards
    "two_fa_enabled", # Binary: second factor active at transaction time
    "email_verified",  # Binary: verified contact channel
    "phone_verified",   # Binary: verified contact channel
    "preferred_channel",  # pos, online, contactless, atm, or swipe
    "loyalty_points_balance",  # Proxy for account tenure and engagement

    # Geography
    "country"  # ISO-2 country of the customer's registered address
]

profile_lookup = FeatureLookup(
    table_name = gold_customer_profile_table,
    feature_names = profile_feature_names,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key
)

# Behavioural aggregations over rolling windows
aggregation_feature_names = [
    # 1-hour window (very short-term velocity)
    "count_tx_1h",  # Number of transactions in the last hour
    "sum_amount_1h",  # Total spend in the last hour
    "avg_amount_1h",  # Average ticket in the last hour
    "distinct_merchants_1h",  # Unique merchants in the last hour
    "count_cross_border_1h",  # Cross-border transactions in the last hour

    # 24-hour window (intra-day behaviour)
    "count_tx_24h",
    "sum_amount_24h",
    "avg_amount_24h",
    "max_amount_24h",  # Largest single transaction in 24 hhours
    "distinct_merchants_24h",
    "distinct_countries_24h",  # Country diversity (spikes signal card cloning)
    "count_tor_vpn_24h",  # Transactions from anonymizing networks
    "count_3ds_failed_24h",  # Failed 3-D secure attempts (strong fraud signal)

    # 7-day window (weekly pattern)
    "count_tx_7d",
    "sum_amount_7d",
    "avg_amount_7d",
    "distinct_merchants_7d",
    "distinct_countries_7d",
    "distinct_devices_7d",  # Device diversity (new device → higher risk)

    # 30-day window (monthly baseline)
    "count_tx_30d",
    "sum_amount_30d",
    "avg_amount_30d",
    "max_amount_30d",
    "min_amount_30d",
    "distinct_merchants_30d",
    "distinct_countries_30d",
    "num_fraud_confirmed_30d",  # Confirmed frauds in the last 30 days
    "spend_24h_vs_avg_30d_ratio"  # 24 hours total spend divided by 30-day average transaction (outlier detector)
]

aggregations_lookup = FeatureLookup(
    table_name = gold_customer_aggregations_table,
    feature_names = aggregation_feature_names,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key
)

# Final list passed
feature_lookups = [profile_lookup, aggregations_lookup]

print(f"Profile features: {len(profile_feature_names)}")
print(f"Aggregation features : {len(aggregation_feature_names)}")
print(f"Total feature columns: {len(profile_feature_names) + len(aggregation_feature_names)}")


## 4. Creación del conjunto de datos de entrenamiento con `create_training_set`

`fe.create_training_set` es la función central de esta libreta. Recibe la *spine*, la lista de `FeatureLookup` y el nombre de la columna que contiene la etiqueta de supervisión, y devuelve un objeto `TrainingSet`.

El `TrainingSet` es un objeto lógico: en este punto todavía no se ha ejecutado ningún cálculo. La `API` construye internamente el plan de ejecución del `PiT` *join*, pero la materialización real en un `DataFrame` de `Spark` no ocurre hasta que se llama a `.load_df()` en la siguiente celda. Esta distinción es relevante porque permite inspeccionar el plan antes de ejecutarlo.

El parámetro `label` le indica a `create_training_set` cuál es la columna objetivo. Esta información queda registrada en los metadatos del `TrainingSet` para que el sistema identifique qué variable es la que se debe predecir.

El parámetro `exclude_columns` permite eliminar columnas que no deben formar parte del conjunto de entrenamiento. En nuestro caso, se excluye `label_available_date` porque es un dato operativo que no se recibiría en una transacción cruda cuando llegue a producción. Descartar este tipo de columnas garantiza que el modelo solo aprenda de la información que verdaderamente tendrá disponible durante la inferencia en tiempo real.

In [0]:
# The column the model must learn to predict.
label = "is_fraud"

# Columns that must be excluded from the final training dataset are those
# that would not be received in a raw transaction when it arrives in production.
# Dropping them ensures the model only trains on data that will actually be
# available during real-time inference.
exclude_columns = ["label_available_date"]

# Build the training dataset logical plan. The following line does not trigger
# any computation. Instead, it returns an object that encodes the spine dataset
# to start from, the feature lookups to join along with their point-in-time
# semantics, the column acting as the label, and the specific columns to drop
# before materialization.
training_dataset = fe.create_training_set(
    df = spine_df,
    feature_lookups = feature_lookups,
    label = label,
    exclude_columns = exclude_columns
)

print("Training dataset logical plan created, but no data materialized yet.")
print(f"Label column: {label}")
print(f"Excluded columns: {exclude_columns}")

In [0]:
# Materialize the training dataset.
# This is the step that actually executes the joins across the cluster.
training_df = training_dataset.load_df()

# Verify the result: total rows and final column set
print(f"Training dataset rows: {training_df.count():,}")
print(f"Training dataset columns: {len(training_df.columns)}")
training_df.printSchema()

In [0]:
# Quick visual inspection of the enriched dataset
training_df.limit(5).toPandas()


## 5. Validación de calidad antes de guardar

Antes de persistir el conjunto de entrenamiento en `Delta`, se realizan tres comprobaciones de calidad sobre el `DataFrame` materializado para detectar posibles anomalías que pudieran invalidar el entrenamiento:

* **Nulos en características**: se contabilizan los valores faltantes por columna. Un volumen inesperado podría indicar problemas en el `PiT` *join* o falta de cobertura histórica en las tablas de la capa `Gold`.
* **Consistencia del volumen**: se verifica que el número de filas del `DataFrame` enriquecido coincida exactamente con el de la *spine* original para asegurar que el cruce temporal no ha provocado pérdida de datos ni duplicidades por productos cartesianos.
* **Balance de clases**: se analiza la distribución de la variable objetivo (`is_fraud`) para cuantificar el desequilibrio natural del fraude y detectar posibles registros que aún carezcan de etiqueta.

In [0]:
feature_columns = [
    column for column in training_df.columns
    if column not in ["customer_id", "transaction_timestamp", "is_fraud"]
]

In [0]:
# Check 1: null count per feature column
null_counts_row = training_df.select([
    count(when(col(column).isNull(), column)).alias(column) for column in feature_columns
]).collect()[0]

for column in feature_columns:
    print(f"{column}: {null_counts_row[column]:,}")

In [0]:
# Check 2: row count consistency
spine_count = spine_df.count()
training_count = training_df.count()

print(f"Spine dataset rows: {spine_count:,}")
print(f"Training dataset rows: {training_count:,}")

In [0]:
# Check 3: class balance
print("Class balance:")
class_balance_rows = (
    training_df.groupBy("is_fraud")
               .count()
               .withColumn(
                   "pct", round(col("count") / training_count * 100, 2)
                )
               .orderBy("is_fraud").collect()
)

for row in class_balance_rows:
    value = "NULL" if row["is_fraud"] is None else row["is_fraud"]
    print(f"Label {value}: {row['count']:,} rows ({row['pct']}%)")


Los resultados obtenidos sobre nuestro conjunto de 60 millones de transacciones arrojan información clave que valida la arquitectura y la naturaleza de los datos:

* **Consistencia del volumen**: el conteo de filas es idéntico al de la *spine* de entrada. Esto confirma que el `PiT` *join* ha funcionado a la perfección y de forma unívoca.
* **Nulos en características**: se observa una presencia masiva de nulos que responde a dos escenarios previstos. Por un lado, las características estáticas del perfil del cliente presentan nulos porque el origen de datos solo proporciona la foto actual sin su historial de cambios; al viajar al pasado, el `PiT` *join* no encuentra un registro válido para esa fecha. Por ejemplo, el cliente `C001294FFE92A` realizó transacciones entre finales de 2022 y principios de 2023, pero su único registro de perfil disponible tiene como fecha de inicio de validez (`__START_AT`) el 8 de febrero de 2025. Al evaluar el cruce `AS OF` la fecha de la transacción, el sistema no encuentra ningún perfil que fuera válido en 2022 o 2023, devolviendo nulo para características como su edad, género o segmento. Por otro lado, los nulos en características comportamentales responden a una realidad matemática: si un cliente no tiene compras en los últimos 30 días, métricas estadísticas como el importe medio o máximo son indefinidas. La arquitectura no falsea la base de datos, sino que delega la imputación de todos estos valores perdidos al *pipeline* de modelado.
* **Balance de clases**: se confirma el fuerte desequilibrio típico en detección de fraude, con un 96.85% de transacciones legítimas y un 2.93% de casos positivos. Este dato es vital para configurar posteriormente los pesos en el algoritmo. Además, se detecta un 0.22% de transacciones con la etiqueta nula; corresponden a operaciones muy recientes que aún no han madurado ni han sido auditadas, por lo que su estado es desconocido y deberán descartarse en la fase de entrenamiento.


## 6. Persistencia en `Delta` y registro de metadatos en `Unity Catalog`

Una vez validado el `DataFrame`, y tras **filtrar aquellas transacciones recientes que aún carecen de etiqueta (`is_fraud` nulo)**, se escribe en `Unity Catalog` como tabla `Delta` estática. El modo `overwrite` reemplaza el contenido de la tabla en cada ejecución, pero **no destruye el historial**: `Delta Lake` conserva todas las versiones anteriores de forma automática, lo que permite recuperar cualquier *snapshot* pasado mediante *time travel*.

Para garantizar una trazabilidad inequívoca entre los datos consumidos y el modelo entrenado, se persisten dos versiones distintas como propiedades de la tabla:

* **Versión semántica** (`ml.delta_semantic_version`): contador controlado por el *pipeline*, incrementado automáticamente en cada regeneración del conjunto de datos. Es la referencia de negocio: `0` es el conjunto de datos inicial, `1` es el primero de producción, etc.
* **Versión física** (`ml.delta_physical_version`): versión interna de `Delta` en el momento exacto del `WRITE`, capturada **antes** de que las operaciones `SET TBLPROPERTIES` y `COMMENT` la incrementen automáticamente. Es esta versión la que se usa para el *time travel* en la libreta de entrenamiento, ya que apunta al *snapshot* de datos exacto independientemente de las operaciones de metadatos posteriores.
* **Fecha máxima de los datos** (`ml.data_max_date`): fecha máxima del campo `timestamp` en el conjunto de datos actual. En el siguiente ciclo de reentrenamiento, la libreta de entrenamiento la leerá como `ml.data_previous_max_date` para anclar el corte de validación sin necesidad de releer los datos históricos.
* **Fecha máxima del ciclo anterior** (`ml.data_previous_max_date`): fecha máxima del ciclo anterior, capturada antes de sobreescribir la tabla. Es la referencia que la libreta de entrenamiento usa para calcular los *splits* automáticamente en producción.

La propiedad `delta.enableChangeDataFeed` se activa en la tabla de salida para que, si en el futuro se configura un *pipeline* de monitorización incremental, pueda leer los cambios acumulados sin releer toda la tabla.

In [0]:
def _get_next_semantic_version(table_name):
    """
    Read the current `ml.delta_version` from the table properties and return
    the next semantic version. Returns `0` if the table does not exist yet
    or the property has not been set.
    """
    if not spark.catalog.tableExists(table_name):
        return 0
    properties_df = spark.sql(f"SHOW TBLPROPERTIES {table_name}")
    version_row = properties_df.filter("key = 'ml.delta_semantic_version'").first()
    return int(version_row["value"]) + 1 if version_row else 0


def _get_current_max_date(table_name, date_column):
    """
    Read `ml.data_max_date` from the current table properties and return it
    as a `String`. Returns `None` if the table does not exist yet or the
    property has not been set. This value becomes `ml.data_previous_max_date`
    in the next cycle.
    """
    if not spark.catalog.tableExists(table_name):
        return None
    properties_df = spark.sql(f"SHOW TBLPROPERTIES {table_name}")
    max_date_row = properties_df.filter("key = 'ml.data_max_date'").first()
    return max_date_row["value"] if max_date_row else None

In [0]:
# Filter out recent transactions that lack a supervision label yet.
# A supervised learning algorithm cannot learn from unlabelled data.
clean_training_df = training_df.filter("is_fraud IS NOT NULL")
clean_training_count = clean_training_df.count()

# Resolve the next semantic version and capture the current maximum date
# before writing so both values reflect the previous cycle.
delta_semantic_version = _get_next_semantic_version(gold_training_dataset_table)
data_previous_max_date = _get_current_max_date(gold_training_dataset_table, "timestamp")

# Persist the enriched training set as a static Delta table.
# Using overwrite so the table always reflects the latest generation run.
# Delta Lake automatically retains all previous versions for time travel.
(
    clean_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(gold_training_dataset_table)
)

# Capture the physical Delta version immediately after the write and
# before any ALTER TABLE increments it. This is the version that must
# be used for time travel in the training notebook to guarantee that
# the read points to the data snapshot and not to a metadata operation.
delta_physical_version = int(
    spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}")
    .select("version")
    .first()[0]
)

# Capture the max date of the new data for use in the next cycle
data_max_date = (
    clean_training_df
    .agg(max(col("timestamp")).alias("max_date"))
    .collect()[0]["max_date"]
    .strftime("%Y-%m-%d")
)

print(f"Training dataset saved to: {gold_training_dataset_table}")
print(f"Semantic version: {delta_semantic_version}")
print(f"Physical version: {delta_physical_version}")
print(f"Data maximum date: {data_max_date}")
print(f"Data previous maximum date: {data_previous_max_date}")

In [0]:
generated_at = datetime.now(timezone.utc).isoformat()

# SET TAGS: surface metadata in the Catalog Explorer for human inspection.
# Tags belong to the data, not to a training run, and are visible
# in the Unity Catalog user interface but cannot be read via SHOW TBLPROPERTIES.
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TAGS (
        'delta_semantic_version' = '{delta_semantic_version}',
        'delta_physical_version' = '{delta_physical_version}',
        'data_max_date' = '{data_max_date}',
        'data_previous_max_date' = '{data_previous_max_date}',
        'spine_table' = '{gold_spine_table}',
        'feature_table_1' = '{gold_customer_profile_table}',
        'feature_table_2' = '{gold_customer_aggregations_table}',
        'label_col' = 'is_fraud',
        'num_rows' = '{clean_training_count}',
        'num_features' = '{len(feature_columns)}',
        'generated_at' = '{generated_at}'
    )
""")

# SET TBLPROPERTIES: persist the same metadata as table properties so that
# the training notebook can resolve the delta version programmatically
# via SHOW TBLPROPERTIES to keep a clean data → model traceability chain.
# The "ml." prefix avoids collisions with internal properties.
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TBLPROPERTIES (
        'ml.delta_semantic_version' = '{delta_semantic_version}',
        'ml.delta_physical_version' = '{delta_physical_version}',
        'ml.data_max_date' = '{data_max_date}',
        'ml.data_previous_max_date' = '{data_previous_max_date}',
        'ml.spine_table' = '{gold_spine_table}',
        'ml.feature_table_1' = '{gold_customer_profile_table}',
        'ml.feature_table_2' = '{gold_customer_aggregations_table}',
        'ml.label_col' = 'is_fraud',
        'ml.num_rows' = '{clean_training_count}',
        'ml.num_features' = '{len(feature_columns)}',
        'ml.generated_at' = '{generated_at}'
    )
""")

# Add a human-readable description to the table
table_description = """
This managed table acts as the **static training dataset** for the machine learning model.
It is a point-in-time (`PiT`) snapshot joining the `gold_fraud_spine` with customer profiles
(`gold_customer_profile`) and rolling-window aggregations (`gold_customer_aggregations`),
pre-filtered to exclude unlabelled transactions.
"""
spark.sql(f"COMMENT ON TABLE {gold_training_dataset_table} IS '{table_description}'")

print(f"Unity Catalog metadata set on: {gold_training_dataset_table}")
print(f"Semantic version: {delta_semantic_version}")
print(f"Physical version: {delta_physical_version}")
print(f"Data maximum date: {data_max_date}")
print(f"Data previous maximum date: {data_previous_max_date}")
print(f"Number of rows: {clean_training_count:,}")
print(f"Number of features: {len(feature_columns)}")
print(f"Generated at: {generated_at}")
print()


## 7. Verificación del historial de versiones `Delta`

La última sección de esta libreta muestra el historial de operaciones de la tabla recién creada y verifica que el *time travel* apunta al *snapshot* de datos correcto.

En el ecosistema `Delta`, cada vez que se sobrescriben los datos o se modifican los metadatos, el motor registra internamente una nueva versión numérica incremental de forma automática. Esta capacidad de *time travel* es el estándar de la industria en entornos `MLOps`, ya que hace que los conjuntos de datos sean completamente inmutables y reproducibles a lo largo del tiempo.

El historial refleja la secuencia exacta de operaciones de esta ejecución:

* **Versión física `N`**: `WRITE`; los datos reales, la única versión relevante para el modelo.
* **Versión física `N + 1`**: `SET TBLPROPERTIES`; los metadatos `ml.*`.
* **Versión física `N + 2`**: `SET TBLPROPERTIES`; el comentario de la tabla.

Por este motivo, `ml.delta_physical_version` se captura entre el `WRITE` y el primer `ALTER TABLE`, garantizando que siempre apunta a los datos y nunca a una operación de metadatos.

In [0]:
# Retrieve and display the internal version history.
# We dynamically extract all available metadata for each commit.
history_df = spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}")
history_rows = history_df.collect()

for row in history_rows:
    for col_name in history_df.columns:
        val = row[col_name]
        # Skip nulls or empty dictionaries to keep the output clean and readable
        if val is not None and val != {} and val != "":
            print(f"{col_name}: {val}")
    print()

In [0]:
try:
    # Retrieve the latest table properties
    properties_df = spark.sql(f"SHOW TBLPROPERTIES {gold_training_dataset_table}")

    # Extract the semantic and physical versions and data maximum and previous maximum date using the "ml." prefix convention
    semantic_version_row = properties_df.filter("key = 'ml.delta_semantic_version'").first()
    physical_version_row = properties_df.filter("key = 'ml.delta_physical_version'").first()
    max_date_row = properties_df.filter("key = 'ml.data_max_date'").first()
    previous_max_date_row = properties_df.filter("key = 'ml.data_previous_max_date'").first()

    delta_semantic_version = int(semantic_version_row["value"]) if semantic_version_row else None
    delta_physical_version = int(physical_version_row["value"]) if physical_version_row else None
    data_max_date = max_date_row["value"] if max_date_row else None
    data_previous_max_date = previous_max_date_row["value"] if previous_max_date_row else None

    print(f"Semantic version: {delta_semantic_version}")
    print(f"Physical version: {delta_physical_version}")
    print(f"Data maximum date: {data_max_date}")
    print(f"Previous maximum date: {data_previous_max_date}")

    # Time travel: load the exact snapshot of data, which points to the exact
    # write snapshot regardless of subsequent ALTER TABLE operations
    df_past = (
        spark.read
             .format("delta")
             .option("versionAsOf", delta_physical_version)
             .table(gold_training_dataset_table)
    )
    print(f"Rows loaded via time travel: {df_past.count():,}")

except Exception as e:
    print(f"Failed to retrieve the version or load the data. Error: {e}")


## 8. Conclusiones y siguientes pasos

### ¿Qué hemos visto?

En esta libreta hemos construido el conjunto de datos de entrenamiento usando la `API` del `Feature Store` de `Databricks`:

1. La *spine* (`gold_fraud_spine`) define qué filas componen el conjunto de datos y en qué instante temporal se evalúa cada cruce. Es el punto de entrada obligatorio de `create_training_set`.
2. Los objetos `FeatureLookup` declaran cómo enriquecer cada fila de la *spine* con características de las tablas `Gold`. El parámetro `timestamp_lookup_key` activa el `PiT` *join*, que garantiza que cada fila recibe únicamente los valores de características disponibles en el momento exacto de la transacción, eliminando la posibilidad de fuga de datos del futuro.
3. `fe.create_training_set` construye el plan de ejecución y `.load_df()` materializa el `PiT` *join* de forma distribuida en `Spark`. El cruce se ejecuta una única vez y el resultado se guarda en `Delta`, de modo que los experimentos de hiperparámetros posteriores solo realizan lecturas secuenciales sobre la tabla estática.
4. Los metadatos del conjunto de datos se registran como *tags* y propiedades en `Unity Catalog`. Para garantizar una trazabilidad inequívoca entre los datos consumidos y el modelo entrenado, se persisten cuatro campos complementarios. La **versión semántica** (`ml.delta_semantic_version`) es un contador de negocio incrementado automáticamente en cada regeneración (`0` es el conjunto de datos inicial del seminario, `1` es el primero de producción, etc.) y es el valor que la libreta de entrenamiento registrará en el sistema de seguimiento de experimentos junto con los hiperparámetros y métricas obtenidas. La **versión física** (`ml.delta_physical_version`) es la versión interna de `Delta` en el momento exacto del `WRITE`, capturada antes de que las operaciones `SET TBLPROPERTIES` y `COMMENT` la incrementen automáticamente, y permite reproducir exactamente el *snapshot* de datos usado en cada ciclo mediante *time travel*. La **fecha máxima de los datos** (`ml.data_max_date`) registra la fecha más reciente del campo `timestamp` en el conjunto de datos actual y en el siguiente ciclo de reentrenamiento se leerá como `ml.data_previous_max_date` para anclar el corte de validación sin necesidad de releer datos históricos. Finalmente, `ml.data_previous_max_date` es la fecha máxima del ciclo anterior, capturada antes de sobreescribir la tabla: el *test window* del siguiente ciclo es todo lo que hay entre esta fecha y `ml.data_max_date`, la validación son los `VALIDATION_WINDOW_MONTHS` anteriores a ella, y el entrenamiento los `TRAINING_WINDOW_MONTHS` anteriores a la validación. Esta es la separación de responsabilidades correcta: los metadatos pertenecen al dato; el sistema de seguimiento de experimentos recibirá `ml.delta_semantic_version` más adelante, cuando ya haya un modelo real al que enlazarlo.
5. Las comprobaciones de calidad (nulos, consistencia de volumen y balance de clases) permiten detectar problemas de cobertura en las tablas `Gold` antes de lanzar un ciclo de entrenamiento.

### ¿Cuándo volver a ejecutar esta libreta?

* **Primera vez**: para generar el conjunto de entrenamiento inicial y lanzar el primer ciclo de experimentación.
* **Cuando el sistema de monitorización detecte *drift* significativo**: el *job* de reentrenamiento automatizado lanzará esta libreta con datos más recientes y sobrescribirá la tabla `Delta`. `Delta Lake` conservará automáticamente la versión anterior en su historial.
* **Cuando se amplíe el conjunto de características**: si se añaden nuevas tablas de características a la capa `Gold`, habrá que actualizar los objetos `FeatureLookup` y regenerar el conjunto de datos.
* **Nunca de forma manual en producción**: esta libreta está diseñada para ejecutarse como tarea dentro de un *job* automatizado (por ejemplo, en `Lakeflow`). La ejecución manual solo tiene sentido durante el desarrollo.

### ¿Qué sigue?

Con el conjunto de entrenamiento disponible en `gold_fraud_training_dataset`, las dos líneas de trabajo que siguen pueden avanzar en paralelo:

1. **Fase de modelado**: con los datos listos, comienza la etapa de creación y entrenamiento de algoritmos. Las libretas de experimentación consumirán directamente la tabla estática `gold_fraud_training_dataset`. Durante estos ciclos, se consultará `ml.delta_physical_version` para cargar mediante *time travel* exactamente el mismo *snapshot* de datos que generó este *pipeline*, y se registrará `ml.delta_semantic_version` en el sistema de seguimiento de experimentos junto con los hiperparámetros probados y las métricas obtenidas. Esto asegurará la trazabilidad exacta entre el conjunto de datos consumido y los distintos modelos entrenados.

2. **Configuración del *baseline* de monitorización**: la tabla estática `gold_fraud_training_dataset` recién generada se usará como perfil de referencia (*baseline*) en la monitorización de calidad de datos. A medida que el modelo opere en producción, las características evaluadas en cada nueva transacción se registrarán en una tabla de inferencia (*inference table*), la cual comparte el mismo esquema exacto que nuestro conjunto de entrenamiento. Mediante la capacidad de `Data profiling`, el sistema calculará automáticamente las estadísticas de estas nuevas instancias y comparará la distribución de las características entrantes con la de nuestro *baseline* estático para identificar posibles desviaciones (*data drift*). Si el *drift* supera el umbral configurado, el sistema generará una alerta. Esta notificación permitirá al equipo evaluar si el cambio en el comportamiento de los datos justifica lanzar de nuevo esta libreta para generar una versión actualizada del conjunto de datos y comenzar un nuevo ciclo de reentrenamiento.